# DV360 Contextual Bandits

In [0]:
# run this installer if contextual bandits is not on your cluster already
%pip install contextualbandits


In [0]:
import pandas as pd
import numpy as np
import mlflow

from pyspark.sql import functions as F

from sklearn.linear_model import LogisticRegression
import contextualbandits
from contextualbandits.online import BootstrappedUCB, BootstrappedTS, LogisticUCB, AdaptiveGreedy, EpsilonGreedy
from copy import deepcopy

## Data Prep

In [0]:
spark_df = spark.sql("""
SELECT
  date,
  country,
  device_type,
  creative,
  clicks,
  impressions
FROM marketing.display_video_360_silver.campaign_metrics
WHERE country IN (
  'IE','PT','PL','BE'
)
""")

In [0]:
from pyspark.sql import functions as F

# Reduce creative column to text before first '_'
spark_df = spark_df.withColumn(
    "creative",
    F.split(F.col("creative"), "_").getItem(0)
)

# Aggregation
spark_agg = (
    spark_df
    .groupBy("date", "country", "device_type", "creative")
    .agg(
        F.sum("impressions").alias("impressions"),
        F.sum("clicks").alias("clicks")
    )
)


In [0]:
df2 = spark_agg.toPandas()

In [0]:
df2.head()

In [0]:
import pandas as pd
import numpy as np

# 1. Build reward (binary)
df2["reward"] = (df2["clicks"] > 0).astype(int)

# 2. Context encoding (one-hot)
context_cols = ["country", "device_type"]
X_context = pd.get_dummies(df2[context_cols], drop_first=False)
X = X_context.values  # feature matrix for bandit

# 3. Arm indexing
df2["arm_idx"] = df2["creative"].astype("category").cat.codes
arm_mapping = dict(zip(df2["arm_idx"], df2["creative"]))

# 4. Reward matrix for bandit
# Each row = one observation
# Each column = one creative/arm
# Multiply dummy for each arm by the reward (0/1)
arm_dummies = pd.get_dummies(df2["arm_idx"], prefix="arm")
y = df2["reward"].values[:, None] * arm_dummies.values

# 5. Sanity checks
print("Context matrix X shape:", X.shape)
print("Reward matrix y shape:", y.shape)
print("Number of creatives (arms):", arm_dummies.shape[1])
print("Number of rows (observations):", len(df2))


In [0]:
df2.iloc[:,:-1].head()

## Full batch models

In [0]:

nchoices = y.shape[1]
base_algorithm = LogisticRegression(solver='lbfgs', warm_start=True)
beta_prior = ((3./nchoices, 4), 2) # until there are at least 2 observations of each class, will use this prior
beta_prior_ucb = ((5./nchoices, 4), 2) # UCB gives higher numbers, thus the higher positive prior
beta_prior_ts = ((2./np.log2(nchoices), 4), 2)
### Important!!! the default values for beta_prior will be changed in version 0.3

## The base algorithm is embedded in different metaheuristics
bootstrapped_ucb = BootstrappedUCB(deepcopy(base_algorithm), nchoices = nchoices,
                                   beta_prior = beta_prior_ucb, percentile = 80,
                                   random_state = 1111)
bootstrapped_ts = BootstrappedTS(deepcopy(base_algorithm), nchoices = nchoices,
                                 beta_prior = beta_prior_ts, random_state = 2222)
epsilon_greedy = EpsilonGreedy(deepcopy(base_algorithm), nchoices = nchoices,
                               beta_prior = beta_prior, random_state = 4444)
logistic_ucb = LogisticUCB(nchoices = nchoices, percentile = 70,
                           beta_prior = beta_prior_ts, random_state = 5555)
adaptive_greedy_thr = AdaptiveGreedy(deepcopy(base_algorithm), nchoices=nchoices,
                                     decay_type='threshold',
                                     beta_prior = beta_prior, random_state = 6666)
adaptive_greedy_perc = AdaptiveGreedy(deepcopy(base_algorithm), nchoices = nchoices,
                                      decay_type='percentile', decay=0.9997,
                                       beta_prior=beta_prior, random_state = 7777)

models = [bootstrapped_ucb, bootstrapped_ts, epsilon_greedy, logistic_ucb]

In [0]:
# These lists will keep track of the rewards obtained by each policy
rewards_ucb, rewards_ts, rewards_egr, rewards_lucb = [list() for i in range(len(models))]

lst_rewards = [rewards_ucb, rewards_ts, rewards_egr, rewards_lucb]

# batch size - algorithms will be refit after N rounds
batch_size = 100

# initial seed - all policies start with the same small random selection of actions/rewards
first_batch = X[:batch_size, :]
np.random.seed(1)
action_chosen = np.random.randint(nchoices, size=batch_size)
rewards_received = y[np.arange(batch_size), action_chosen]

# fitting models for the first time
for model in models:
    model.fit(X=first_batch, a=action_chosen, r=rewards_received)
    
# these lists will keep track of which actions does each policy choose
lst_a_ucb, lst_a_ts, lst_a_egr, lst_a_lucb  = [action_chosen.copy() for i in range(len(models))]

lst_actions = [lst_a_ucb, lst_a_ts, lst_a_egr, lst_a_lucb]

# rounds are simulated from the full dataset
def simulate_rounds(model, rewards, actions_hist, X_global, y_global, batch_st, batch_end):
    np.random.seed(batch_st)
    
    ## choosing actions for this batch
    # selects a creative for each input row from X_global
    actions_this_batch = model.predict(X_global[batch_st:batch_end, :]).astype('uint8')
    
    # keeping track of the sum of rewards received
    # 
    rewards.append(y_global[np.arange(batch_st, batch_end), actions_this_batch].sum())
    
    # adding this batch to the history of selected actions
    new_actions_hist = np.append(actions_hist, actions_this_batch)
    
    # now refitting the algorithms after observing these new rewards
    np.random.seed(batch_st)
    model.fit(X_global[:batch_end, :], new_actions_hist, y_global[np.arange(batch_end), new_actions_hist])
    
    return new_actions_hist

In [0]:
# Disable autologging so each .fit() doesn't create a new run
mlflow.autolog(disable=True)

# now running all the simulation
for i in range(int(np.floor(X.shape[0] / batch_size))):
    batch_st = (i + 1) * batch_size
    batch_end = (i + 2) * batch_size
    batch_end = np.min([batch_end, X.shape[0]])
    
    for model in range(len(models)):
        lst_actions[model] = simulate_rounds(models[model],
                                             lst_rewards[model],
                                             lst_actions[model],
                                             X, y,
                                             batch_st, batch_end)

In [0]:
import matplotlib.pyplot as plt
from pylab import rcParams
%matplotlib inline


def get_mean_reward(reward_lst, batch_size=batch_size):
    mean_rew = list()
    for r in range(len(reward_lst)):
        mean_rew.append(sum(reward_lst[:r+1]) * 1.0 / ((r+1)*batch_size))
    return mean_rew

# Plot settings
rcParams['figure.figsize'] = 25, 15
lwd = 5
cmap = plt.get_cmap('tab20')
colors = plt.cm.tab20(np.linspace(0, 1, 20))

ax = plt.subplot(111)

# Replace rewards_ucb, rewards_ts, etc. with your own reward lists
plt.plot(get_mean_reward(rewards_ucb), label="Upper Confidence Bound", linewidth=lwd, color=colors[0])
plt.plot(get_mean_reward(rewards_ts), label="Thompson Sampling", linewidth=lwd, color=colors[2])
plt.plot(get_mean_reward(rewards_egr), label="Epsilon-Greedy", linewidth=lwd, color=colors[6])
plt.plot(get_mean_reward(rewards_lucb), label="Logistic UCB", linewidth=lwd, color=colors[8])

# Legend settings
box = ax.get_position()
ax.set_position([box.x0, box.y0 + box.height * 0.1,
                 box.width, box.height * 1.25])
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05),
          fancybox=True, ncol=3, prop={'size':20})

# Axis settings
plt.tick_params(axis='both', which='major', labelsize=25)
plt.xticks([i*20 for i in range(15)], [i*1000 for i in range(15)])

# Labels and title
plt.xlabel('Rounds (models were updated every 500 rounds)', size=30)
plt.ylabel('Cumulative Mean Reward', size=30)
plt.title('Comparison of Contextual Bandit Policies\n(Custom Dataset)\n\nCreative Level Data', size=30)
plt.grid()
plt.show()

next steps:
- understand plot better
- write more detailed notes for each step in the training loop
- understand difference between batch and full and why it's good